### joins 

In [0]:
df = spark.createDataFrame(
    [("iPhone", 999), ("Samsung", 799), ("Mouse", 20)],
    ["product", "price"]
)


In [0]:
def segment(price):
    return "Premium" if price > 500 else "Normal"

udf_segment = udf(segment)

df.withColumn("customer_type", udf_segment("price"))


DataFrame[product: string, price: bigint]


In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

def segment(price):
    return "Premium" if price > 500 else "Normal"

udf_segment = udf(segment, StringType())

df2 = df.withColumn("customer_type", udf_segment("price"))
df2.show(truncate=False)


+-------+-----+-------------+
|product|price|customer_type|
+-------+-----+-------------+
|iPhone |999  |Premium      |
|Samsung|799  |Premium      |
|Mouse  |20   |Normal       |
+-------+-----+-------------+



In [0]:
events = (
    spark.read
    .option("header","true")
    .option("inferSchema","true")
    .csv("/Volumes/workspace/ecommerce/ecommerce_data/2019-Oct.csv")
)
events.count()
events.show(5, truncate=False)


+-------------------+----------+----------+-------------------+-----------------------------------+--------+-------+---------+------------------------------------+
|event_time         |event_type|product_id|category_id        |category_code                      |brand   |price  |user_id  |user_session                        |
+-------------------+----------+----------+-------------------+-----------------------------------+--------+-------+---------+------------------------------------+
|2019-10-01 00:00:00|view      |44600062  |2103807459595387724|NULL                               |shiseido|35.79  |541312140|72d76fde-8bb3-4e00-8c23-a032dfed738c|
|2019-10-01 00:00:00|view      |3900821   |2053013552326770905|appliances.environment.water_heater|aqua    |33.2   |554748717|9333dfbd-b87a-4708-9857-6336556b0fcc|
|2019-10-01 00:00:01|view      |17200506  |2053013559792632471|furniture.living_room.sofa         |NULL    |543.1  |519107250|566511c2-e2e3-422b-b695-cf8e6e792ca8|
|2019-10-01 00:0

In [0]:
products = spark.createDataFrame(
    [(1001, "Phone"), (1002, "Laptop")],
    ["product_id", "product_name"]
)

joined = events.join(products, on="product_id", how="left")
joined.select("product_id", "product_name").show(5)


+----------+------------+
|product_id|product_name|
+----------+------------+
|  44600062|        NULL|
|   3900821|        NULL|
|  17200506|        NULL|
|   1307067|        NULL|
|   1004237|        NULL|
+----------+------------+
only showing top 5 rows


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, sum as Fsum

w = Window.partitionBy("user_id").orderBy(col("event_time"))

events_rt = events.withColumn("running_total",
                              Fsum(col("price")).over(w))

events_rt.select("user_id","event_time","price","running_total").show(5, truncate=False)


+---------+-------------------+-------+-------------+
|user_id  |event_time         |price  |running_total|
+---------+-------------------+-------+-------------+
|205053188|2019-10-09 10:30:19|162.17 |162.17       |
|205053188|2019-10-09 10:30:44|162.17 |324.34       |
|209714031|2019-10-20 18:29:45|1686.02|1686.02      |
|209714031|2019-10-20 18:30:08|75.16  |1761.18      |
|209714031|2019-10-24 18:18:25|1686.02|3447.2       |
+---------+-------------------+-------+-------------+
only showing top 5 rows


In [0]:
from pyspark.sql.functions import when

features = (events
    .withColumn("price_bucket",
        when(col("price") >= 500, "High")
        .when(col("price") >= 100, "Medium")
        .otherwise("Low")
    )
    .withColumn("is_premium",
        when(col("price") >= 500, 1).otherwise(0)
    )
)

features.select("price","price_bucket","is_premium").show(5)


+-------+------------+----------+
|  price|price_bucket|is_premium|
+-------+------------+----------+
|  35.79|         Low|         0|
|   33.2|         Low|         0|
|  543.1|        High|         1|
| 251.74|      Medium|         0|
|1081.98|        High|         1|
+-------+------------+----------+
only showing top 5 rows
